# AstraSim Network Performance Analysis

This notebook analyzes and compares network performance data from AstraSim simulation logs across different network configurations and congestion awareness methods.

## Overview

We'll analyze the network logs from different configurations:
- Network topologies: FullyConnected_, Ring_, and Switch
- Methods: Congestion-aware and congestion-unaware

We'll create visualizations to compare:
- Network topologies
- Performance metrics (latency, delay, bandwidth)
- Communication patterns over time
- Effects of congestion

In [ ]:
# Import required libraries
import os
import glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx
from collections import defaultdict
import re
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

## 1. Data Loading and Preprocessing

First, we'll create functions to load and parse the network log files from different configurations.

In [ ]:
# Define the base directory where all log files are stored
base_dir = "/home/xavid/feina/astra-sim/upc/comparing_networks/output/"

# Function to parse the AstraSim log file
def parse_astrasim_log(file_path):
    # Skip first line which is the header info line
    with open(file_path, 'r') as f:
        lines = f.readlines()
    
    # Extract the header
    header_line = lines[0].strip()
    header_parts = header_line.split(',')
    header_parts = [part.strip() for part in header_parts]
    
    # Parse data lines
    data = []
    for line in lines[1:]:
        # Extract the CSV part of the line (after the log prefix)
        csv_part = line.split(':', 1)[1].strip() if ':' in line else line.strip()
        parts = csv_part.split(',')
        parts = [part.strip() for part in parts]
        
        if len(parts) >= len(header_parts):
            data.append(parts[:len(header_parts)])
    
    # Create DataFrame
    df = pd.DataFrame(data, columns=header_parts)
    
    # Convert numeric columns to appropriate types
    numeric_cols = ['src', 'dst', 'tensor_size', 'tag', 'workload_node_id', 
                    'issue_tick', 'bandwidth', 'dims_count', 'topology', 
                    'hops', 'latency', 'delay']
    
    for col in numeric_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')
    
    return df

# Function to find all log files and organize them by collective and network type
def collect_log_files():
    log_files = {}
    
    # Walk through the base directory
    for root, dirs, files in os.walk(base_dir):
        for file in files:
            if file.endswith('astrasim.log_network.csv'):
                full_path = os.path.join(root, file)
                
                # Extract collective type from path
                path_parts = full_path.split('/')
                collective_type = path_parts[-3] if len(path_parts) >= 3 else "unknown"
                
                # Extract network type
                network_type = path_parts[-2] if len(path_parts) >= 2 else "unknown"
                
                # Determine if congestion aware or unaware
                is_congestion_aware = True
                if "_Unaware" in full_path:
                    is_congestion_aware = False
                
                # Create nested dictionary structure
                if collective_type not in log_files:
                    log_files[collective_type] = {}
                

                key = f"{network_type.replace('_', '').replace('Unaware', '')}_{'congestion_aware' if is_congestion_aware else 'congestion_unaware'}"
                log_files[collective_type][key] = full_path
    
    return log_files

# Load all log files
log_files_dict = collect_log_files()

# Print found files for verification
for collective, networks in log_files_dict.items():
    print(f"Collective: {collective}")
    for network, file_path in networks.items():
        print(f"  - {network}: {file_path}")

# Load a sample file to check the structure
sample_files = []
for collective, networks in log_files_dict.items():
    for network, file_path in networks.items():
        sample_files.append((collective, network, file_path))
        if len(sample_files) >= 2:  # Limit to 2 samples
            break
    if len(sample_files) >= 2:
        break

# Load and show sample data
if sample_files:
    collective, network, file_path = sample_files[0]
    print(f"\nSample data from {collective} - {network}:")
    sample_df = parse_astrasim_log(file_path)
    display(sample_df.head())

In [ ]:
# Function to load all data into a structured dictionary
def load_all_data():
    all_data = {}
    for collective, networks in log_files_dict.items():
        all_data[collective] = {}
        for network, file_path in networks.items():
            try:
                df = parse_astrasim_log(file_path)
                all_data[collective][network] = df
                print(f"Loaded {collective} - {network}: {len(df)} rows")
            except Exception as e:
                print(f"Error loading {file_path}: {e}")
    return all_data

# Load all data
all_data = load_all_data()

In [ ]:
# Function to compare congestion approaches focusing only on issue_tick and delay
def compare_congestion_methods(all_data):
    """
    Compare congestion-aware vs unaware approaches for the same network types,
    focusing only on issue_tick and delay.
    """
    # Store results for summary
    comparison_results = []
    
    # Process each collective
    for collective, networks in all_data.items():
        print(f"\n=== Analyzing {collective} ===")
        
        # Group networks by type (removing the congestion part from the name)
        network_types = {}
        for network_name in networks.keys():
            if '_congestion_aware' in network_name:
                network_type = network_name.replace('_congestion_aware', '')
                if network_type not in network_types:
                    network_types[network_type] = {'aware': None, 'unaware': None}
                network_types[network_type]['aware'] = network_name
            elif '_congestion_unaware' in network_name:
                network_type = network_name.replace('_congestion_unaware', '')
                if network_type not in network_types:
                    network_types[network_type] = {'aware': None, 'unaware': None}
                network_types[network_type]['unaware'] = network_name
        
        # Compare each network type
        for network_type, config in network_types.items():
            if config['aware'] and config['unaware']:
                df_aware = networks[config['aware']]
                df_unaware = networks[config['unaware']]
                
                # Calculate basic statistics for issue_tick and delay
                aware_issue_mean = df_aware['issue_tick'].mean()
                unaware_issue_mean = df_unaware['issue_tick'].mean()
                aware_delay_mean = df_aware['delay'].mean()
                unaware_delay_mean = df_unaware['delay'].mean()
                
                # Calculate percentage differences
                issue_diff_pct = ((unaware_issue_mean - aware_issue_mean) / unaware_issue_mean * 100 
                                  if unaware_issue_mean != 0 else 0)
                delay_diff_pct = ((unaware_delay_mean - aware_delay_mean) / unaware_delay_mean * 100 
                                  if unaware_delay_mean != 0 else 0)
                
                # Determine if there's a significant difference (more than 1%)
                has_issue_diff = abs(issue_diff_pct) > 1
                has_delay_diff = abs(delay_diff_pct) > 1
                has_difference = has_issue_diff or has_delay_diff
                
                # Save results
                comparison_results.append({
                    'collective': collective,
                    'network_type': network_type,
                    'aware_issue_mean': aware_issue_mean,
                    'unaware_issue_mean': unaware_issue_mean,
                    'issue_diff_pct': issue_diff_pct,
                    'aware_delay_mean': aware_delay_mean,
                    'unaware_delay_mean': unaware_delay_mean,
                    'delay_diff_pct': delay_diff_pct,
                    'has_difference': has_difference
                })
                
                # Print if there's a difference or not
                if has_difference:
                    print(f"\n{network_type}: Differences detected between congestion methods")
                    if has_issue_diff:
                        print(f"  Issue Tick: {issue_diff_pct:.2f}% difference (Aware: {aware_issue_mean:.2f}, Unaware: {unaware_issue_mean:.2f})")
                    if has_delay_diff:
                        print(f"  Delay: {delay_diff_pct:.2f}% difference (Aware: {aware_delay_mean:.2f}, Unaware: {unaware_delay_mean:.2f})")
                    
                    # Create visualizations for configurations with differences
                    create_comparison_plots(df_aware, df_unaware, collective, network_type)
                else:
                    print(f"\n{network_type}: No significant differences between congestion methods")
            else:
                print(f"\n{network_type}: Missing aware or unaware configuration")
    
    return pd.DataFrame(comparison_results)

# Function to create comparative plots
def create_comparison_plots(df_aware, df_unaware, collective, network_type):
    """
    Create visualizations comparing issue_tick and delay between
    congestion-aware and congestion-unaware configurations.
    """
    # Create a figure with 2 rows and 2 columns
    fig, axes = plt.subplots(2, 2, figsize=(18, 12))
    
    # 1. Distribution of issue_ticks
    sns.histplot(df_aware['issue_tick'], kde=True, ax=axes[0, 0], 
                 label='Congestion Aware', color='blue', alpha=0.5)
    sns.histplot(df_unaware['issue_tick'], kde=True, ax=axes[0, 0], 
                 label='Congestion Unaware', color='red', alpha=0.5)
    axes[0, 0].set_title(f'Distribution of Issue Ticks - {network_type}')
    axes[0, 0].set_xlabel('Issue Tick')
    axes[0, 0].set_ylabel('Count')
    axes[0, 0].legend()
    
    # 2. Distribution of delays
    sns.histplot(df_aware['delay'], kde=True, ax=axes[0, 1], 
                 label='Congestion Aware', color='blue', alpha=0.5)
    sns.histplot(df_unaware['delay'], kde=True, ax=axes[0, 1], 
                 label='Congestion Unaware', color='red', alpha=0.5)
    axes[0, 1].set_title(f'Distribution of Delays - {network_type}')
    axes[0, 1].set_xlabel('Delay')
    axes[0, 1].set_ylabel('Count')
    axes[0, 1].legend()
    
    # 3. Cumulative message count over time (issue_tick)
    df_aware_sorted = df_aware.sort_values('issue_tick')
    df_unaware_sorted = df_unaware.sort_values('issue_tick')
    
    axes[1, 0].plot(df_aware_sorted['issue_tick'], range(1, len(df_aware_sorted) + 1), 
                  label='Congestion Aware', color='blue', linewidth=2)
    axes[1, 0].plot(df_unaware_sorted['issue_tick'], range(1, len(df_unaware_sorted) + 1), 
                  label='Congestion Unaware', color='red', linewidth=2)
    axes[1, 0].set_title(f'Cumulative Messages by Issue Tick - {network_type}')
    axes[1, 0].set_xlabel('Issue Tick')
    axes[1, 0].set_ylabel('Cumulative Message Count')
    axes[1, 0].legend()
    
    # 4. Scatter plot of issue_tick vs delay
    axes[1, 1].scatter(df_aware['issue_tick'], df_aware['delay'], 
                      label='Congestion Aware', color='blue', alpha=0.5)
    axes[1, 1].scatter(df_unaware['issue_tick'], df_unaware['delay'], 
                      label='Congestion Unaware', color='red', alpha=0.5)
    axes[1, 1].set_title(f'Issue Tick vs Delay - {network_type}')
    axes[1, 1].set_xlabel('Issue Tick')
    axes[1, 1].set_ylabel('Delay')
    axes[1, 1].legend()
    
    plt.suptitle(f'Comparison of Congestion Methods for {collective} - {network_type}', fontsize=16)
    plt.tight_layout()
    plt.subplots_adjust(top=0.93)
    plt.show()

# Function to analyze message timing patterns
def analyze_message_patterns(df_aware, df_unaware, collective, network_type):
    """
    Analyze and visualize message timing patterns between congestion-aware
    and congestion-unaware configurations.
    """
    # Calculate completion time (issue_tick + delay)
    df_aware['completion_time'] = df_aware['issue_tick'] + df_aware['delay']
    df_unaware['completion_time'] = df_unaware['issue_tick'] + df_unaware['delay']
    
    # Get the maximum time for both datasets
    max_time = max(df_aware['completion_time'].max(), df_unaware['completion_time'].max())
    
    # Create time bins
    bin_size = max_time / 20  # 20 bins
    df_aware['time_bin'] = (df_aware['issue_tick'] / bin_size).astype(int)
    df_unaware['time_bin'] = (df_unaware['issue_tick'] / bin_size).astype(int)
    
    # Count messages per time bin
    aware_counts = df_aware.groupby('time_bin').size().reset_index(name='count')
    unaware_counts = df_unaware.groupby('time_bin').size().reset_index(name='count')
    
    # Create visualization
    plt.figure(figsize=(14, 6))
    plt.bar(aware_counts['time_bin'] * bin_size, aware_counts['count'], 
            width=bin_size*0.4, alpha=0.6, label='Congestion Aware', color='blue')
    plt.bar(unaware_counts['time_bin'] * bin_size + bin_size*0.4, unaware_counts['count'], 
            width=bin_size*0.4, alpha=0.6, label='Congestion Unaware', color='red')
    
    plt.title(f'Message Timing Pattern - {collective} - {network_type}')
    plt.xlabel('Time (issue_tick)')
    plt.ylabel('Number of Messages')
    plt.legend()
    plt.grid(alpha=0.3)
    plt.show()
    
    # Create a heatmap of node communication over time
    plt.figure(figsize=(15, 6))
    
    # Create subplots
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 8))
    
    # For congestion aware
    for _, row in df_aware.sample(min(1000, len(df_aware))).iterrows():  # Sample to avoid overcrowding
        ax1.plot([row['issue_tick'], row['issue_tick'] + row['delay']], 
                [row['src'], row['dst']], 'b-', alpha=0.2)
    
    ax1.set_title(f'Congestion Aware - {network_type}')
    ax1.set_xlabel('Time')
    ax1.set_ylabel('Node ID')
    ax1.grid(alpha=0.3)
    
    # For congestion unaware
    for _, row in df_unaware.sample(min(1000, len(df_unaware))).iterrows():
        ax2.plot([row['issue_tick'], row['issue_tick'] + row['delay']], 
                [row['src'], row['dst']], 'r-', alpha=0.2)
    
    ax2.set_title(f'Congestion Unaware - {network_type}')
    ax2.set_xlabel('Time')
    ax2.set_ylabel('Node ID')
    ax2.grid(alpha=0.3)
    
    plt.suptitle(f'Message Flow Visualization - {collective}', fontsize=16)
    plt.tight_layout()
    plt.subplots_adjust(top=0.9)
    plt.show()

# Function to create a summary table
def create_summary_table(comparison_results):
    """
    Create a summary table of all comparison results.
    """
    # Filter to show only configurations with differences
    diff_results = comparison_results[comparison_results['has_difference']]
    no_diff_results = comparison_results[~comparison_results['has_difference']]
    
    print("\n=== Configurations with NO significant differences ===")
    if len(no_diff_results) == 0:
        print("None - All configurations show differences")
    else:
        for _, row in no_diff_results.iterrows():
            print(f"- {row['collective']} - {row['network_type']}")
    
    print("\n=== Configurations with significant differences ===")
    if len(diff_results) == 0:
        print("None - No configurations show differences")
    else:
        # Create a formatted table
        summary_data = []
        for _, row in diff_results.iterrows():
            summary_data.append({
                'Collective': row['collective'],
                'Network Type': row['network_type'],
                'Issue Tick Diff %': f"{row['issue_diff_pct']:.2f}%",
                'Delay Diff %': f"{row['delay_diff_pct']:.2f}%",
                'Aware Issue Mean': f"{row['aware_issue_mean']:.2f}",
                'Unaware Issue Mean': f"{row['unaware_issue_mean']:.2f}",
                'Aware Delay Mean': f"{row['aware_delay_mean']:.2f}",
                'Unaware Delay Mean': f"{row['unaware_delay_mean']:.2f}",
            })
        
        summary_df = pd.DataFrame(summary_data)
        display(summary_df)

# Run the analysis
comparison_results = compare_congestion_methods(all_data)

# Create summary table
create_summary_table(comparison_results)

# For detailed analysis of configurations with differences
for _, row in comparison_results[comparison_results['has_difference']].iterrows():
    collective = row['collective']
    network_type = row['network_type']
    
    # Get the dataframes
    aware_key = f"{network_type}_congestion_aware"
    unaware_key = f"{network_type}_congestion_unaware"
    
    df_aware = all_data[collective][aware_key]
    df_unaware = all_data[collective][unaware_key]
    
    # Analyze message patterns
    analyze_message_patterns(df_aware, df_unaware, collective, network_type)

## 2. Network Topology Visualization

Next, we'll visualize the network topologies to understand the communication patterns in each configuration.

In [ ]:
# Function to create network visualizations and save them locally
def visualize_and_save_networks(all_data, output_dir='network_visualizations'):
    # Create output directory if it doesn't exist
    import os
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
        print(f"Created directory: {output_dir}")
    
    # Store figure references
    network_figures = {}
    saved_paths = []
    
    # Counter to limit to just 4 figures for the grid
    figure_count = 0
    max_figures = 10
    
    # Process data and create visualizations
    for collective, networks in all_data.items():
        network_figures[collective] = {}
        
        for network_name, df in networks.items():
            # Create network graph
            G = create_network_graph(df)
            title = f"{collective} - {network_name}"
            
            # Create and save the figure
            fig = visualize_network(G, title)
            
            # Save figure to file
            filename = f"{collective}_{network_name.replace('/', '_')}.png"
            filepath = os.path.join(output_dir, filename)
            fig.savefig(filepath, dpi=300, bbox_inches='tight')
            print(f"Saved figure to: {filepath}")
            
            # Store references
            network_figures[collective][network_name] = fig
            
            # Track for grid display (only first 4)
            if figure_count < max_figures:
                saved_paths.append({
                    'path': filepath,
                    'title': title,
                    'fig': fig
                })
                figure_count += 1
            
            # Close the individual figure to free memory
            plt.close(fig)
            
            # If we have enough figures, we can stop processing more
            if figure_count >= max_figures:
                break
        
        # If we have enough figures, we can stop processing more
        if figure_count >= max_figures:
            break
    
    # Create a 2x2 grid of miniature figures
    create_network_grid(saved_paths, output_dir)
    
    return network_figures

# Function to create a 2x2 grid of network visualizations
def create_network_grid(saved_figures, output_dir):
    # Create a 2x2 grid
    fig, axes = plt.subplots(2, 2, figsize=(12, 10))
    axes = axes.flatten()
    
    # Add each saved figure to the grid
    for i, fig_data in enumerate(saved_figures[:4]):  # Limit to first 4
        # Get the original figure
        orig_fig = fig_data['fig']
        title = fig_data['title']
        
        # Create a simplified version of the network
        img = plt.imread(fig_data['path'])
        axes[i].imshow(img)
        axes[i].set_title(title, fontsize=10)
        axes[i].axis('off')
    
    # Adjust layout and save
    plt.tight_layout()
    grid_path = os.path.join(output_dir, 'network_grid.png')
    plt.savefig(grid_path, dpi=300, bbox_inches='tight')
    print(f"Saved grid visualization to: {grid_path}")
    plt.show()

# Run the visualization and saving
if all_data:
    network_figures = visualize_and_save_networks(all_data)

In [ ]:
# Compare congestion-aware vs congestion-unaware topologies
def compare_network_topologies(collective):
    if collective not in all_data:
        print(f"No data found for collective: {collective}")
        return
    
    networks = all_data[collective]
    
    # Get all network types (FullyConnected_, Ring_, Switch)
    network_types = set()
    for network_name in networks.keys():
        network_type = network_name.split('_congestion')[0]
        network_types.add(network_type)
    
    for network_type in network_types:
        aware_key = f"{network_type}_congestion_aware"
        unaware_key = f"{network_type}_congestion_unaware"
        
        if aware_key in networks and unaware_key in networks:
            fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 8))
            
            # Create graphs
            G_aware = create_network_graph(networks[aware_key])
            G_unaware = create_network_graph(networks[unaware_key])
            
            # Use the same layout for both graphs for better comparison
            pos = nx.spring_layout(nx.compose(G_aware, G_unaware), seed=42)
            
            # Draw congestion-aware graph
            plt.sca(ax1)
            nx.draw_networkx_nodes(G_aware, pos, node_size=500, node_color='skyblue')
            nx.draw_networkx_edges(G_aware, pos, alpha=0.7, edge_color='blue', arrowsize=15)
            nx.draw_networkx_labels(G_aware, pos, font_size=12)
            ax1.set_title(f"{collective} - {network_type} (Congestion Aware)")
            ax1.axis('off')
            
            # Draw congestion-unaware graph
            plt.sca(ax2)
            nx.draw_networkx_nodes(G_unaware, pos, node_size=500, node_color='lightgreen')
            nx.draw_networkx_edges(G_unaware, pos, alpha=0.7, edge_color='green', arrowsize=15)
            nx.draw_networkx_labels(G_unaware, pos, font_size=12)
            ax2.set_title(f"{collective} - {network_type} (Congestion Unaware)")
            ax2.axis('off')
            
            plt.tight_layout()
            plt.show()
        else:
            print(f"Missing data for comparison of {network_type} in {collective}")

# Compare topologies for a selected collective
if all_data:
    selected_collective = list(all_data.keys())[0]
    compare_network_topologies(selected_collective)